# Phase 2 — Real Data Ingestion (run in Colab, CPU-only)

Builds the actual pretraining corpus: streams General English (FineWeb-Edu + OpenWebText) and
the natural-dialogue sources from HuggingFace, tokenizes, and writes resumable shard files to
Google Drive, tracked by `bina_pretrain/data_manifest.py`. **No GPU needed** — use a
CPU-only / free-tier Colab runtime for this, so the A100/L4 compute-unit budget stays reserved
for Phase 3's actual training. Run this notebook top to bottom once per session, however many
sessions it takes; every run resumes exactly where the last one left off, tracked in
`manifest_ingest.json` on Drive — a source is never re-downloaded or re-tokenized from scratch.

See `pretrain/README.md` and `configs/data_mix.yaml` for the real, measured source list and
target token counts (revised from the original plan after real verification —
`bigscience/open_subtitles_monolingual` doesn't exist, see the comment at the top of
`data_mix.yaml` for the full story).

## 1. Setup

Same Colab-detection pattern as `phase3_pretrain_training.ipynb`: mounts Drive in real Colab,
falls back to local paths with a clear notice if run locally instead (e.g. for testing).
`datasets`/`pyyaml`/`tiktoken` are the only packages not already in the Colab image.

In [ ]:
!pip install -q datasets pyyaml tiktoken

import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/Bina_Pretrain"
    REPO_URL = "https://github.com/YENOSven/gpt-bina.git"
    CODE_DIR = "/content/gpt-bina"
    if os.path.exists(CODE_DIR):
        subprocess.run(["git", "-C", CODE_DIR, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, CODE_DIR], check=True)
else:
    print("NOT running inside Google Colab -- falling back to local paths. This only tests "
          "the notebook's own logic; a real multi-session ingestion run needs an actual Colab "
          "session so Drive persists shard progress across sessions your PC won't be on for.")

    def _find_repo_root():
        for candidate in (os.getcwd(), os.path.join(os.getcwd(), "..", ".."), os.path.join(os.getcwd(), "..")):
            candidate = os.path.abspath(candidate)
            if os.path.isdir(os.path.join(candidate, "pretrain", "bina_pretrain")):
                return candidate
        raise RuntimeError("couldn't find the repo root automatically -- set CODE_DIR manually and re-run")

    CODE_DIR = _find_repo_root()
    DRIVE_ROOT = os.path.join(CODE_DIR, "pretrain", "_local_drive_stub")

os.makedirs(DRIVE_ROOT, exist_ok=True)
sys.path.insert(0, f"{CODE_DIR}/pretrain")
print(f"code ready at {CODE_DIR}, Drive root at {DRIVE_ROOT}")

# HuggingFace rate-limits anonymous IPs on repeated load_dataset() calls (hit this for real
# during Phase 2 development). Set HF_TOKEN (Colab: Settings -> Secrets, or paste directly
# below) for a much higher limit on a real multi-session ingestion run.
HF_TOKEN = ""  # optional: paste a token here, or leave blank
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN


## 2. Session config

`SESSION_MINUTES` bounds how long THIS session spends ingesting — keep it well inside your
actual Colab session length so the notebook always stops itself cleanly rather than getting
killed mid-write. Everything else is read straight from `configs/data_mix.yaml`, the single
source of truth for what to ingest and how much of it — not duplicated here.

In [ ]:
import time

from bina_pretrain import data_manifest as dm
from bina_pretrain import data_pipeline as dp

SESSION_MINUTES = 180  # 3 hours; adjust to whatever this session's realistic budget is
STOP_MARGIN_MINUTES = 5
CHUNK_TOKENS = 10_000_000     # tokens requested per ingest_source() call
TOKENS_PER_SHARD = 10_000_000  # tokens per shard file on disk

MANIFEST_PATH = f"{DRIVE_ROOT}/manifest_ingest.json"
SHARD_DIR = f"{DRIVE_ROOT}/shards"
MIX_YAML_PATH = f"{CODE_DIR}/pretrain/configs/data_mix.yaml"

mix_config = dp.load_mix_config(MIX_YAML_PATH)
manifest = dm.load(MANIFEST_PATH)
dp.init_manifest_from_mix(manifest, mix_config)  # safe to call every session -- never resets progress
dm.save(manifest, MANIFEST_PATH)

source_names = dp.all_source_names(mix_config)
print(f"{len(source_names)} sources tracked: {source_names}")


## 3. Ingest

Works through sources in order, giving each one `CHUNK_TOKENS` at a time and saving the
manifest to Drive after every chunk (so an interruption loses at most one chunk's progress,
never more). Moves to the next source once the current one is complete or genuinely exhausted
(ran out of real data before reaching its target -- expected for the small natural-dialogue
sources, which are being pulled close to their full real size). Stops when `SESSION_MINUTES`
runs out, whether or not everything's done -- that's normal, just run this notebook again
another day.

In [ ]:
session_deadline = time.time() + (SESSION_MINUTES - STOP_MARGIN_MINUTES) * 60

for source_name in source_names:
    if time.time() >= session_deadline:
        print(f"\nsession time budget reached, stopping before {source_name}")
        break

    entry = manifest[source_name]
    if dm.is_complete(entry) or entry["status"] == dm.STATUS_EXHAUSTED:
        print(f"{source_name}: already {entry['status']}, skipping")
        continue

    print(f"\ningesting {source_name} (phase={entry['phase']}, targets={entry['targets']})...")
    while not dm.is_complete(entry) and entry["status"] != dm.STATUS_EXHAUSTED and time.time() < session_deadline:
        tokens = dp.ingest_source(manifest, source_name, SHARD_DIR, TOKENS_PER_SHARD, max_tokens_this_call=CHUNK_TOKENS)
        dm.save(manifest, MANIFEST_PATH)
        print(f"  +{tokens:,} tokens -> ingested={entry['tokens_ingested']}, "
              f"phase={entry['phase']}, status={entry['status']}")
        if tokens == 0:
            break

print("\ndone for this session.")


## 4. Status

In [ ]:
print(f"{'source':30s} {'status':12s} {'train':>14s} {'val':>10s} {'test':>10s}")
for name in source_names:
    e = manifest[name]
    t = e["tokens_ingested"]
    print(f"{name:30s} {e['status']:12s} {t['train']:>14,} {t['val']:>10,} {t['test']:>10,}")

all_done = all(manifest[n]["status"] in (dm.STATUS_COMPLETE, dm.STATUS_EXHAUSTED) for n in source_names)
print(f"\nall sources done (complete or exhausted): {all_done}")


## 5. Combine into the final corpus (only once every source is done)

One-time step, not resumable (re-run from scratch if interrupted -- it's fast relative to
ingestion): shuffles every source's documents together per split and writes the final
`corpus_train.bin` / `corpus_val.bin` / `corpus_test.bin` that Phase 3's training actually
reads. Skips cleanly with a message if sources are still in progress.

In [ ]:
if all_done:
    for phase in ("train", "val", "test"):
        out_path = f"{DRIVE_ROOT}/corpus_{phase}.bin"
        n_tokens = dp.combine_shards_to_corpus(manifest, source_names, SHARD_DIR, phase, out_path)
        print(f"{phase}: {n_tokens:,} tokens -> {out_path}")
    print("\nCorpus ready. Phase 3's training notebook can now point CORPUS_PATH at corpus_train.bin.")
else:
    remaining = [n for n in source_names if manifest[n]["status"] not in (dm.STATUS_COMPLETE, dm.STATUS_EXHAUSTED)]
    print(f"not all sources done yet ({len(remaining)} remaining: {remaining}) -- run more sessions first.")
